# Task 2 — Agent A with memory

Ask in plain language (`what is the news for apple`, `latest AAPL price`, or the full assessment prompt).

Agent A reads your question, decides which tools to call (sometimes 1, sometimes all 5), and writes a `DataBrief`.

Follow-ups reuse `logs/session.json` and the disk tool cache in `logs/cache/` so the same facts are not refetched.

Keys: local `.env`, Colab Secrets. Never paste a key into a cell.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REPO_URL = os.environ.get(
    "AGENTIC_WORKFLOW_REPO",
    "https://github.com/lavanblavan/Agentic_financial_Analyser.git",
)

def ensure_project() -> Path:
    if IN_COLAB:
        root = Path("/content/Agentic_financial_Analyser")
        if not (root / "src/config.py").exists():
            subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
        else:
            subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=False)
        os.chdir(root)
    else:
        root = Path.cwd().resolve()
        if not (root / "src/config.py").exists():
            raise FileNotFoundError("Open this notebook from the Agentic_financial_Analyser repo root.")

    req = root / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root

ROOT = ensure_project()
print("project root:", ROOT)
print("runtime:", "colab" if IN_COLAB else "local")

In [ ]:
from src.config import describe_env, load_settings, missing_key_help

settings = load_settings()
print(describe_env(settings))
if not settings.llm_ready:
    print(missing_key_help(settings))

In [ ]:
# Messy question, bare name/ticker, or the full assessment prompt.
QUERY = "what is the news for apple"

from src.task_profile import infer_task_profile
from src.ticker import parse_research_query

parsed = parse_research_query(QUERY)
profile = infer_task_profile(QUERY)
COMPANY = parsed["ticker"]
print("question:", parsed["task"])
print("task_mode:", profile["mode"])
print("required tools:", profile["required"])
print(
    f"{parsed['subject']!r} → {parsed['ticker']}  "
    f"({parsed['name']}, via {parsed['resolved_via']}, extract {parsed['extracted_via']})"
)

## Agent A graph

`START → agent → tools → agent → … → finalize`

The LLM picks tool order. The graph only nudges for tools required by your question.

In [ ]:
from src.agent_a import build_agent_a

agent_a = build_agent_a()
print(agent_a.get_graph().draw_mermaid())

## Run with memory

`ask_agent_a` compares the new question to the last one. Same issuer → reuse the stored brief; tools run only for the gap.

In [ ]:
from src.agent_a import format_answer
from src.memory import ask_agent_a, describe_memory, relate

print("session:", describe_memory())
print("plan:", relate(QUERY))
result = ask_agent_a(QUERY)

print("from_memory:", result.get("from_memory"))
print("task_mode:", result.get("task_mode"))
print("tool call order:")
for i, step in enumerate(result.get("tool_calls") or [], start=1):
    print(f"  {i}. {step['tool']}  {step.get('args') or step}")
if result.get("missing_after_run"):
    print("still missing:", result["missing_after_run"])
print()
print(format_answer(result))

## Follow-up

Ask a related question on the same issuer. Memory answers from the brief or fetches only what is missing.

In [ ]:
FOLLOWUP = "What is the latest apple price?"

print("plan:", relate(FOLLOWUP))
follow = ask_agent_a(FOLLOWUP)
print("from_memory:", follow.get("from_memory"))
print("need_tools:", follow.get("need_tools"))
print("reused:", follow.get("reused"))
print()
print(follow.get("followup_answer") or format_answer(follow))
print()
print("session:", describe_memory())

## Trace log

Tool calls append to `logs/agent_trace.jsonl`.

In [ ]:
import pandas as pd
from src.tracing import read_trace

trace = read_trace(limit=20)
frame = pd.DataFrame(trace)
cols = [c for c in ["ts", "tool", "ok", "cached", "duration_ms", "inputs", "output"] if c in frame.columns]
display(frame[cols] if cols else frame)